In [1]:
import pandas as pd
import json
import yaml
import seaborn as sns
from pathlib import Path

In [2]:
folder = Path('..', 'outputs', '003.experiment')

experiments = [ p for p in folder.iterdir()
                if p.is_dir() and Path(p, 'eval_results.json').exists() ]

for p in experiments: print(p.name)

3dfe76a5d16b4eb3f46cbe0ced4a0efc
81ea8352477b74933d9c6463aac8d779
8b5b67cad31e307f08ceb7cbeb927a3b
ea46ba8c6fb4ce7655c1665782b49e50
99b702e682402d9953e20e473954a4c1
68c231b06262ecb78125237a8e982a0c


In [3]:
def get_params(dir):
    params_file = Path(dir, 'parameters.yaml')
    parameters = yaml.safe_load(params_file.read_text())
    # parameters['folder'] = dir.name
    return parameters

def read_json_results(folder, base_file):
    file = Path(folder, base_file)
    results = json.loads(file.read_text())
    results['folder'] = folder.name
    return results

df_params = pd.DataFrame([ get_params(p) for p in experiments ])
df_eval = pd.DataFrame([read_json_results(e, 'eval_results.json') for e in experiments])
df_valids = pd.DataFrame([read_json_results(e, 'valid_results.json') for e in experiments])

df = pd.merge(df_params, df_eval, left_on='_hash_id', right_on='folder')

teacher_keys = df['teachers_keys'].loc[0] # ['t5', 'llama']

df = pd.concat([
    df,
    df['teachers_weights'].apply(pd.Series, index=teacher_keys)
], axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   _hash_id                 6 non-null      object 
 1   _timestamp               6 non-null      object 
 2   batch_size               6 non-null      int64  
 3   bf16                     6 non-null      bool   
 4   dataset                  6 non-null      object 
 5   eval_steps               6 non-null      int64  
 6   experiment               6 non-null      object 
 7   from_pretrained          6 non-null      object 
 8   generation_max_length    6 non-null      int64  
 9   grad_steps               6 non-null      int64  
 10  local_rank               6 non-null      int64  
 11  logging_strategy         6 non-null      object 
 12  lora_rank                5 non-null      float64
 13  lr                       6 non-null      float64
 14  max_input_length         6 non

In [4]:
df_valids

,epoch,eval_accuracy,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_token_accuracy,folder
0,15.053763,0.752,0.012954,43.5707,11.476,0.367,0.82336,3dfe76a5d16b4eb3f46cbe0ced4a0efc
1,15.053763,0.760,0.015109,38.6896,12.923,0.414,0.82354,81ea8352477b74933d9c6463aac8d779
2,15.053763,0.768,0.014632,39.7894,12.566,0.402,0.82352,8b5b67cad31e307f08ceb7cbeb927a3b
3,15.053763,0.760,0.015109,39.3737,12.699,0.406,0.82354,ea46ba8c6fb4ce7655c1665782b49e50
4,15.053763,0.760,0.015451,40.5984,12.316,0.394,0.82334,99b702e682402d9953e20e473954a4c1
5,15.053763,0.756,0.013669,40.8984,12.225,0.391,0.82346,68c231b06262ecb78125237a8e982a0c


In [6]:
df[
    ['dataset', '_timestamp', 'batch_size', 'lr', 'lora_rank', 'eval_accuracy', 'eval_token_accuracy', 'eval_loss']
].sort_values(by='eval_accuracy', ascending=False).fillna(16)

,dataset,_timestamp,batch_size,lr,lora_rank,eval_accuracy,eval_token_accuracy,eval_loss
5,obqa,2025-11-16T10:27:42.946974,32,0.0005,256.0,0.712,0.91770,0.018116
2,obqa,2025-11-15T23:19:10.285503,32,0.0005,128.0,0.712,0.91760,0.019200
1,obqa,2025-11-14T14:07:19.770734,32,0.0005,16.0,0.708,0.91780,0.020906
0,obqa,2025-11-16T18:20:30.965050,32,0.0005,512.0,0.708,0.91790,0.016267
3,obqa,2025-11-15T10:36:43.739195,32,0.0005,32.0,0.708,0.91780,0.020906
4,obqa,2025-11-15T17:24:50.245816,32,0.0005,64.0,0.690,0.91788,0.021196
